# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Real-world framing:** this is FlyRank's own operating problem. The client portfolio this dataset is drawn from carries hundreds of thousands of live search pages, and only a small fraction can get editorial attention in any given cycle — so the work is deciding *which* fraction.

**Research question:** What distinct performance archetypes exist across the content inventory, and what action does each archetype suggest?

**Lane:** 3 — Structured Content Archetype Clustering (unsupervised). Metric & metadata clustering only — never semantic/text.

**Unit of analysis:** one content item (page), described by safe 90-day-observable search-side signals plus metadata.

**Output → action:** a small set of named archetypes, each mapped to one editorial action — protect / improve / rewrite / merge / prune / monitor — so a content lead can route a large page inventory by archetype instead of triaging page by page.

**Cost of a wrong call:** mistakes land in batches, not one page at a time. Pruning a hidden gem loses traffic permanently; spending rewrite capacity on noise wastes editorial time across every page sharing that label.

**Why ML, not a fixed rule:** the strongest pairwise correlation among the candidate signals is 0.33 (log-impressions × word count) — no pair is redundant enough to collapse into a single if-statement axis, so a fixed threshold can't recover the structure a multi-dimensional clustering finds.

In [1]:
# Re-stated from work/notebooks/w01_research_question.ipynb and w02_ml_task_framing.ipynb.
# Heavy intermediates are gitignored (see .gitignore), so this cell hardcodes published
# aggregates rather than recomputing from data/ -- the re-run path for these numbers is
# w03_data_contract.ipynb -> w05_model.ipynb -> w06_validation_audit.ipynb.

framing = {
    "task_type": "clustering (unsupervised)",
    "unit_of_analysis": "one content item (page)",
    "target_or_proxy": "none -- cluster assignment is the closest thing to a proxy",
    "strongest_pairwise_corr_among_candidate_features": 0.33,  # log_impressions x word_count, w02
    "silhouette_untuned_k5_n21109": 0.278,                      # w01 provisional run, k not yet tuned
}
for k, v in framing.items():
    print(f"{k}: {v}")

assert 0 <= framing["strongest_pairwise_corr_among_candidate_features"] <= 1
assert 0 <= framing["silhouette_untuned_k5_n21109"] <= 1

task_type: clustering (unsupervised)
unit_of_analysis: one content item (page)
target_or_proxy: none -- cluster assignment is the closest thing to a proxy
strongest_pairwise_corr_among_candidate_features: 0.33
silhouette_untuned_k5_n21109: 0.278


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** Hugging Face `hf://datasets/FlyRank/internship-warehouse`.

**Tables used:**
- `fact_content_daily_performance`, partition `month=2026-03` (the development month — the `_sample` table, a sealed later month, is never touched, to avoid peeking at the outcome window)
- `dim_content.parquet` — metadata: `word_count`, `content_type`, `main_intent`
- `dim_clients` — read only for `gsc_data_start` / `ga4_data_start` availability checks, never as a feature

**Date window:** `report_date` between 2026-03-01 and 2026-03-31 (31 days).

**Grain:** one row = one content item. The fact table (9,841,378 rows, 331,437 distinct `content_hash_id`) is aggregated to one row per item and joined to `dim_content` (0 duplicate ids on either side).

**What was excluded, and why:**

| Excluded | Why |
|---|---|
| `content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id` | pseudonymous join/group keys — no ordinal meaning; using one as a feature would let a model memorize identity instead of learning from measured behavior |
| `engagement_rate` | proven leak in a held-out probe — honest-feature ROC AUC 0.856 climbs to 0.877 once this is added, because it's engineered directly from the label source (`engaged_sessions`) |
| `ga4_*` columns where `ga4_data_available = FALSE` | zero-filled placeholders, not real zero engagement; row-level GA4 coverage is only 4.2% True (65.1% False, 30.7% NULL), and even content-level only 27.3% of pages carry any real GA4 history |
| `trend_direction`, `trend_pct` | label-derived — the exact columns a supervised target would be built from |
| `content_updated_date` | live/mutable dimension (242 distinct values, max date beyond the whole data window) with no snapshot version in this release |
| `last_optimized_date` / `optimization_eligible_date` | decision-derived and post-March; only 8.7% of items carry it, in a partial rollout |

**Public-safety note:** no client names, domains, URLs, page titles, or raw queries exist anywhere in this release's schema — see `DATA_USE.md`.

In [2]:
# From work/notebooks/w03_data_contract.ipynb (published, re-derivable by re-running that
# notebook against the same partition).
data_contract = {
    "fact_rows_march": 9_841_378,
    "distinct_content_march": 331_437,
    "dim_content_duplicate_ids": 0,
    "row_level_ga4_available_true_pct": 4.2,
    "row_level_ga4_available_false_pct": 65.1,
    "row_level_ga4_available_null_pct": 30.7,
    "content_level_any_ga4_coverage_pct": 27.3,
}
for k, v in data_contract.items():
    print(f"{k}: {v}")

assert abs(
    data_contract["row_level_ga4_available_true_pct"]
    + data_contract["row_level_ga4_available_false_pct"]
    + data_contract["row_level_ga4_available_null_pct"] - 100.0
) < 0.01

fact_rows_march: 9841378
distinct_content_march: 331437
dim_content_duplicate_ids: 0
row_level_ga4_available_true_pct: 4.2
row_level_ga4_available_false_pct: 65.1
row_level_ga4_available_null_pct: 30.7
content_level_any_ga4_coverage_pct: 27.3


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Features (4, all `month=2026-03` search-side signals only):** `log_impressions`, `log_clicks`, `ctr`, `avg_position`.

**What was tried and dropped first.** An earlier feature set added `word_count`, `has_word_count`, `has_ga4_coverage`, and one-hot `content_type`/`main_intent`. It produced a k=2 split that a single depth-1 rule on `has_word_count` reproduced at 99.17% accuracy — a data-availability artifact (word_count missing for ~30% of pages, median-filled at ~2,780, manufacturing a fake gap), not an archetype. `engaged_sessions` was dropped for the same reason: one resulting cluster sat at exactly 1.000 GA4 coverage — it was finding the 27.3% GA4-covered slice, not a behavioral pattern.

**Method:** plain K-Means on standardized features (seed 42). The scaler is re-fit inside each training fold only (`StandardScaler().fit(X.iloc[tr])`, never on the full dataset), so no fold's test statistics leak into its own standardization. PCA is used only as a 2-D inspection projection, never fed to the fit.

**k selection:** swept k=2..10 inside each fold's training clients, rejecting any k whose smallest cluster falls under 2% of pages. k=4 wins on mean silhouette (0.387, about 1.7 SD clear of the runner-up) among the k that pass the guard. 0.387 is a moderate score, not a strong one — the four groups are a real but overlapping structure, not cleanly separated clusters, which is part of why they're treated as a review lens rather than a certainty (see Limitations).

**Validation design — GroupKFold by client, not by time.** There is no future to leak into here (single-month clustering); the risk is a page's own client leaking across train/test. A single 80/20 grouped split was rejected first: across 5 seeds the held-out page share swung from 0.5% to 29.6%, and at seed 42 the held-out set was 100% one content type vs 70% in train — a single split is not a stable estimate. 5-fold GroupKFold is used instead, with out-of-fold cluster labels aligned back to a reference labeling via Hungarian matching on centroid distance.

**Success definition (for the outcome-window comparison in Results):** a page "improved" if its April CTR exceeded its March CTR, requiring April impressions ≥ 100. A first version of this definition (April CTR reaches the March median CTR of its own position tier) was dropped after discovery that two position tiers have a March median CTR of exactly 0.000, which inflates the trivial base rate to 51.5% — kept only as a secondary check restricted to tiers with a positive bar.

**Baseline:** the ML-07 rule — flag a page if it clears a 100-impression volume floor and its CTR sits below the peer median CTR for its position tier; `score = impressions × (peer_median_ctr − ctr) / 100`. Reproduced here bit-for-bit against the modeling population: 101,409 shared rows, score agreement 1.0000. That 101,409 is the 331,437 raw March content items after the same 100-impression volume floor plus a complete-feature requirement (all four search-side features present); the remaining ~69% is mostly pages below the volume floor, which is also why the ranked queue in Recommendations only ever reaches a minority of the raw May inventory.

**Standardization.** Reported O/E ratios use indirect standardization: the expected improve-rate for a group is the population's improve-rate re-weighted to that group's March-CTR-stratum composition (the same 9 strata plotted in the confound chart), so a group isn't credited or penalized for simply containing more low- or high-CTR pages than average.

**Leakage checks:** the model's 4 features are all `month=2026-03` GSC aggregates — no decision-derived, post-period, or identifier column entered the feature list (see the Data section table). A positive-control probe (predicting "improved" from honest features vs. honest + the leaked outcome CTR) scores AUC 0.637 honest vs. 0.940 leaked — confirming the evaluation pipeline can detect a real leak when one is deliberately introduced.

**Multiple comparisons.** Six groups are checked against chance in Results (the rule, four archetypes, and one archetype subset) without a Bonferroni or FDR correction. This is reported as exploratory, decision-support evidence, not a confirmatory hypothesis test — which is also why the ranked recommendations weight durability alongside O/E rather than treating any single CI as proof.

In [3]:
# From work/notebooks/w05_model.ipynb (k-selection sweep) and w06_validation_audit.ipynb
# (positive control). Re-run path: the k-selection cell in w05_model.ipynb.
k_sweep = {
    2: (0.3677, True), 3: (0.3660, True), 4: (0.3874, True),
    5: (0.3235, True), 6: (0.3385, True), 7: (0.3427, False),
    8: (0.3273, False), 9: (0.3202, False), 10: (0.3161, False),
}
eligible = {k: sil for k, (sil, ok) in k_sweep.items() if ok}
best_k = max(eligible, key=eligible.get)
print("k sweep (silhouette, passes >=2% min-cluster guard):")
for k, (sil, ok) in k_sweep.items():
    print(f"  k={k}: silhouette={sil:.4f} guard_pass={ok}")
print(f"selected k = {best_k} (silhouette={eligible[best_k]:.4f})")

positive_control = {"honest_auc": 0.6366, "leaked_auc": 0.9396, "base_rate": 0.2925}
gap = positive_control["leaked_auc"] - positive_control["honest_auc"]
print(f"\npositive control gap (leaked - honest): {gap:.4f}  (pass threshold: > 0.15)")

assert gap > 0.15
assert best_k == 4

k sweep (silhouette, passes >=2% min-cluster guard):
  k=2: silhouette=0.3677 guard_pass=True
  k=3: silhouette=0.3660 guard_pass=True
  k=4: silhouette=0.3874 guard_pass=True
  k=5: silhouette=0.3235 guard_pass=True
  k=6: silhouette=0.3385 guard_pass=True
  k=7: silhouette=0.3427 guard_pass=False
  k=8: silhouette=0.3273 guard_pass=False
  k=9: silhouette=0.3202 guard_pass=False
  k=10: silhouette=0.3161 guard_pass=False
selected k = 4 (silhouette=0.3874)

positive control gap (leaked - honest): 0.3030  (pass threshold: > 0.15)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Raw lift looks real.** ML-07 rule flags (`snippet_fix`) improve at 33.4% vs. a 29.3% base improve-rate across all eligible pages — a 1.14× lift.

**It disappears once March CTR is controlled for.** Using indirect standardization (observed / expected improve-rate, 95% CI bootstrapped BY CLIENT, B=300) against the base rate: rule O/E = 1.03, 95% CI [0.92, 1.16] — chance. Improve-rate falls from ~50% down to ~17% across March-CTR strata on its own, independent of any rule or archetype (nearly monotonic, one small reversal within noise) — the rule was mostly picking out low-CTR pages, and low-CTR pages regress toward a higher improve-rate regardless of any label.

**The clustering doesn't rescue it either.** Headline, stated plainly: once starting CTR is controlled for, neither the ML-07 rule nor three of the four archetype groups demonstrate any skill at predicting April CTR movement.

**Full O/E table (all six groups checked against chance — see the Methodology "multiple comparisons" note, no correction applied):**

| Group | n | O/E | 95% CI | Clears chance? |
|---|---|---|---|---|
| Buried | 9,663 | 0.645 | [0.456, 0.810] | Below 1.0 |
| Overlooked, top-1000 by centroid distance | 1,000 | 0.768 | [0.640, 0.912] | Below 1.0 |
| Overlooked (full group) | 5,715 | 0.896 | [0.714, 1.050] | No — crosses 1.0 |
| Rule: snippet_fix (ML-07) | 36,704 | 1.032 | [0.925, 1.164] | No — crosses 1.0 |
| Long tail | 42,911 | 1.041 | [0.930, 1.150] | No — crosses 1.0 |
| Steady performers | 30,185 | 1.059 | [0.865, 1.233] | No — crosses 1.0 |

**Two findings survive, both pointing DOWN, not up — but they don't carry equal weight:**
- `Buried` archetype, the whole group (9,663 pages): O/E = 0.645, 95% CI [0.456, 0.810] — independently recomputed in the validation audit as 0.667, CI [0.470, 0.855]. Both intervals sit entirely below 1.0. This is the paper's primary finding.
- `Overlooked`, restricted *after the fact* to its top-1000 pages by centroid distance: O/E = 0.768, CI [0.640, 0.912]. The full Overlooked group does **not** clear chance on its own (O/E 0.896, CI [0.714, 1.050]); the top-1000 cut was chosen after seeing that result, to ask whether the most archetype-typical pages behave differently — it is a secondary, post-hoc observation, not independent confirmation, and it is not corrected for testing six groups. It supports "re-verify," not "act now."

**Buried is also the most durable label.** 82.4% of March-labeled Buried pages are still labeled Buried in May, against a 20.7% chance rate under independent relabeling — the most durable of the four archetypes. That combination — a below-chance outcome AND a durable label — is what earns Buried the only Tier-1 (act-now) recommendation; nothing else in the archetype set has both signals agreeing.

**The rule and the model disagree sharply.** The ML-07 rule never once flags an Overlooked page (0% of its queue) — the rule's "CTR below peer median" logic and the clustering's "CTR well above peers" profile for Overlooked pages point in opposite directions by construction.

In [4]:
# From work/notebooks/w05_model.ipynb sections (a)-(d). Bootstrap CIs are client-clustered,
# B=300, seed 42. Re-run path: w05_model.ipynb -> w06_validation_audit.ipynb (independent
# recompute of the Buried O/E, checked separately below).

oe_table = {
    # group: (n, O/E, CI_lo, CI_hi)
    "RULE: snippet_fix (ML-07)":         (36_704, 1.0319, 0.9248, 1.1642),
    "MODEL: Overlooked (improve/expand)": (5_715,  0.8956, 0.7137, 1.0499),
    "MODEL: Steady performers (protect)": (30_185, 1.0590, 0.8653, 1.2332),
    "MODEL: Long tail (monitor)":         (42_911, 1.0409, 0.9296, 1.1501),
    "MODEL: Buried (prune/rewrite)":      (9_663,  0.6452, 0.4560, 0.8099),
    "MODEL: Overlooked top-1000 by dist": (1_000,  0.7675, 0.6402, 0.9115),
}
print(f"{'group':38s} {'n':>8s} {'O/E':>7s} {'CI_lo':>7s} {'CI_hi':>7s}  clears_chance?")
for group, (n, oe, lo, hi) in oe_table.items():
    verdict = "BELOW 1.0" if hi < 1.0 else ("ABOVE 1.0" if lo > 1.0 else "no (crosses 1.0)")
    print(f"{group:38s} {n:8d} {oe:7.4f} {lo:7.4f} {hi:7.4f}  {verdict}")

# ML-09 independent recompute of Buried O/E -- second path, agrees within noise
buried_oe_w05 = 0.645
buried_oe_w09_recompute = 0.667
assert abs(buried_oe_w05 - buried_oe_w09_recompute) < 0.05

# Regression-to-the-mean confound: improve-rate by March CTR stratum. Not strictly
# monotone (one small reversal, 0.2205 -> 0.2217, within noise) but the overall trend
# from the first non-degenerate stratum to the last is a clear decrease.
ctr_strata_improve_rate = [0.2609, 0.4960, 0.4013, 0.3508, 0.3105, 0.2868, 0.2205, 0.2217, 0.1694]
decreasing_pairs = sum(
    1 for a, b in zip(ctr_strata_improve_rate[1:], ctr_strata_improve_rate[2:]) if a >= b
)
print(f"\nstrata pairs decreasing (from the 2nd stratum on): {decreasing_pairs}/{len(ctr_strata_improve_rate) - 2}")
assert ctr_strata_improve_rate[1] > ctr_strata_improve_rate[-1]

# Retention: Buried is the most durable label
retention = {"Steady performers": 0.648, "Long tail": 0.671, "Buried": 0.824, "Overlooked": 0.353}
print("March->May retention by archetype:", retention)
assert retention["Buried"] == max(retention.values())

group                                         n     O/E   CI_lo   CI_hi  clears_chance?
RULE: snippet_fix (ML-07)                 36704  1.0319  0.9248  1.1642  no (crosses 1.0)
MODEL: Overlooked (improve/expand)         5715  0.8956  0.7137  1.0499  no (crosses 1.0)
MODEL: Steady performers (protect)        30185  1.0590  0.8653  1.2332  no (crosses 1.0)
MODEL: Long tail (monitor)                42911  1.0409  0.9296  1.1501  no (crosses 1.0)
MODEL: Buried (prune/rewrite)              9663  0.6452  0.4560  0.8099  BELOW 1.0
MODEL: Overlooked top-1000 by dist         1000  0.7675  0.6402  0.9115  BELOW 1.0

strata pairs decreasing (from the 2nd stratum on): 6/7
March->May retention by archetype: {'Steady performers': 0.648, 'Long tail': 0.671, 'Buried': 0.824, 'Overlooked': 0.353}


## 5. Limitations

*What this work cannot claim.*

- **No causal claim.** Every number here is an observed association in one non-intervention window (March → April), not the effect of taking any action. "O/E below 1.0" means the group's improve-rate came in lower than a CTR-adjusted expectation, not that the label caused pages to get worse.
- **Uneven attrition biases the comparison.** 99.8% of Steady-performer pages survive to the April volume floor vs. 77.3% of Buried pages — the true Buried result is probably worse than the 0.645 reported, since the pages that drop out of the floor are disproportionately the already-weakest ones.
- **44 clients, one at 21.3% of the population.** Wide confidence intervals and a real risk that patterns are one large client's behavior wearing an archetype's name — the GroupKFold-by-client design exists specifically to guard against this, but it doesn't eliminate the concentration.
- **GA4 (on-site behavior) is excluded entirely.** Only 27.3% of pages carry real GA4 history; every archetype here describes search-side behavior only, never what happens after the click.
- **No "ranking lags demand" archetype exists in this feature set.** An earlier run surfaced a 5th cluster that looked like exactly that, but it vanished once `engaged_sessions` was dropped for GA4-coverage leakage — it was a measurement artifact, not a real page type.
- **A small burst-then-revert data-quality pattern exists in the raw signal** (0.74% of the scored queue has `avg_position < 1.0`, consistent with automated crawling/rank-tracking activity, not organic ranking) — flagged, not filtered, since it wasn't material to the results.
- **Clusters are a lens, not a label.** They are a way to group pages for review, not a certified fact about any one page — the no-go list in Ranked recommendations exists because of this.
- **Coverage is partial.** The scored action queue reaches 24.1% of the raw May inventory; 75.9% falls outside scope (unpublished/deleted, no impressions or position data, or below the volume floor).

In [5]:
# From work/notebooks/w06_validation_audit.ipynb, check 5 (population-selection audit).
survival_to_april_floor = {
    "Steady performers": 0.9984, "Long tail": 0.8234, "Buried": 0.7726, "Overlooked": 0.8690,
}
worst_case_vs_published_improve_rate = {
    # group: (worst_case, published, gap)
    "ALL eligible":      (0.2552, 0.2925, -0.0373),
    "Steady performers": (0.3268, 0.3273, -0.0005),
    "Long tail":         (0.2553, 0.3100, -0.0547),
    "Buried":            (0.1471, 0.1904, -0.0433),
    "Overlooked":        (0.1283, 0.1477, -0.0193),
}
print("Survival to April volume floor (higher = less attrition bias):")
for k, v in survival_to_april_floor.items():
    print(f"  {k}: {v:.4f}")

print("\nWorst-case (non-survivors counted as not-improved) vs. published improve-rate:")
for k, (wc, pub, gap) in worst_case_vs_published_improve_rate.items():
    assert abs((wc - pub) - gap) < 1e-4
    print(f"  {k}: worst_case={wc:.4f} published={pub:.4f} gap={gap:+.4f}")

# nothing reorders under the worst-case reweighting
order_published = sorted(worst_case_vs_published_improve_rate,
                          key=lambda k: worst_case_vs_published_improve_rate[k][1])
order_worst_case = sorted(worst_case_vs_published_improve_rate,
                           key=lambda k: worst_case_vs_published_improve_rate[k][0])
assert order_published == order_worst_case
print("\nranking is unchanged under the worst case:", order_published)

Survival to April volume floor (higher = less attrition bias):
  Steady performers: 0.9984
  Long tail: 0.8234
  Buried: 0.7726
  Overlooked: 0.8690

Worst-case (non-survivors counted as not-improved) vs. published improve-rate:
  ALL eligible: worst_case=0.2552 published=0.2925 gap=-0.0373
  Steady performers: worst_case=0.3268 published=0.3273 gap=-0.0005
  Long tail: worst_case=0.2553 published=0.3100 gap=-0.0547
  Buried: worst_case=0.1471 published=0.1904 gap=-0.0433
  Overlooked: worst_case=0.1283 published=0.1477 gap=-0.0193

ranking is unchanged under the worst case: ['Overlooked', 'Buried', 'ALL eligible', 'Long tail', 'Steady performers']


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Tiered by **evidence × durability** — not by raw lift, which Results shows can't be trusted alone.

| Tier | Archetype | Action | Why | Pages (May) |
|---|---|---|---|---|
| **1 — act now** | Buried | prune / rewrite | O/E 0.645–0.667 (CI excludes 1.0, two independent runs) AND 82.4% durable — the only archetype where both signals agree | 20,764 |
| **2 — re-verify before acting** | Overlooked | improve / expand | O/E crosses 1.0 (no clear effect) AND only 35.3% durable — queue position reflects the label's instability, not a case against ever improving these pages | 7,041 |
| **3 — protect** | Steady performers | protect / monitor | O/E at chance — no evidence further action helps — but highest median impressions/CTR, so real value is at stake if neglected | 25,110 |
| **4 — light monitor** | Long tail | monitor | O/E at chance, near-zero median CTR — lowest value at stake | 40,813 |

**Review load:** capped at the top 20 pages per client per review cycle (covers 31.2% of Tier-1 clients fully at 2.4% of Tier-1's page count; the alternative, top-50, only lifts full coverage to 43.8% of clients at more than double the review load).

**No-go list — never automate from a tier alone:** no auto-delete, auto-unpublish, or auto-rewrite without a named human signing off, citing the reason code; never act on Tier 2 without re-checking the page's current archetype; never treat one row's tier assignment as certain; never use this queue for cross-client resourcing without accounting for the 21.3% single-client concentration.

**Retrain triggers** (midpoint of the archetype's own baseline and its chance retention rate): overall retention below 49.4%; Buried below 51.6% (Tier 1's whole justification rests on this number); Overlooked below 21.3%; Long tail below 55.0%; Steady performers below 46.9%. Also: coverage-funnel drift from 24.1%, client-concentration drift from 21.3%, or Buried's O/E confidence interval crossing 1.0 on a fresh re-run.

**Clustering as a foundation, not a dead end.** Most archetypes show no raw O/E lift — read narrowly, a null result for clustering as a scoring tool. But Buried's below-chance O/E *and* 82.4% label durability together show cluster membership itself carries real signal, even where the lift alone doesn't. The honest reading isn't "abandon clustering," it's that this group-level lens is a plausible routing layer for something more targeted later: per-archetype sub-scoring (ranking pages *within* an archetype, not just between them) is a natural next lane, out of scope here and named on purpose in `w07_action_playbook.ipynb`'s Section 3.

In [6]:
# From work/notebooks/w07_action_playbook.ipynb -- the ranked action queue's tier definitions
# and monitoring thresholds. Full queue (93,728 rows) lives at work/outputs/ranked_action_queue.csv
# (gitignored -- regenerate by re-running w05 -> w06 -> w07).
tiers = {
    1: {"archetype": "Buried",            "action": "prune/rewrite",   "n": 20_764, "reason": "buried_worse_than_chance_durable"},
    2: {"archetype": "Overlooked",        "action": "improve/expand",  "n": 7_041,  "reason": "overlooked_no_lift_unstable_reverify"},
    3: {"archetype": "Steady performers", "action": "protect/monitor", "n": 25_110, "reason": "steady_high_value_no_lift"},
    4: {"archetype": "Long tail",         "action": "monitor",         "n": 40_813, "reason": "long_tail_low_value_no_lift"},
}
queue_eligible_total = sum(t["n"] for t in tiers.values())
may_raw_rows = 389_153
print(f"queue-eligible total: {queue_eligible_total} ({queue_eligible_total / may_raw_rows:.1%} of {may_raw_rows} raw May rows)")
for tier_n, t in tiers.items():
    print(f"  Tier {tier_n}: {t['archetype']:20s} -> {t['action']:16s} n={t['n']:6d}  reason={t['reason']}")
assert queue_eligible_total == 93_728

retrain_triggers = {
    "overall": 0.494, "Buried": 0.516, "Overlooked": 0.213, "Long tail": 0.550, "Steady performers": 0.469,
}
current_retention = {
    "overall": 0.658, "Buried": 0.824, "Overlooked": 0.353, "Long tail": 0.671, "Steady performers": 0.648,
}
print("\nretrain-trigger check (current vs. floor):")
for name, floor in retrain_triggers.items():
    current = current_retention[name]
    status = "OK (above floor)" if current >= floor else "RETRAIN"
    print(f"  {name:20s} current={current:.3f} floor={floor:.3f}  {status}")
    assert current >= floor  # true as of the last audit (w07); re-check on every fresh re-run

queue-eligible total: 93728 (24.1% of 389153 raw May rows)
  Tier 1: Buried               -> prune/rewrite    n= 20764  reason=buried_worse_than_chance_durable
  Tier 2: Overlooked           -> improve/expand   n=  7041  reason=overlooked_no_lift_unstable_reverify
  Tier 3: Steady performers    -> protect/monitor  n= 25110  reason=steady_high_value_no_lift
  Tier 4: Long tail            -> monitor          n= 40813  reason=long_tail_low_value_no_lift

retrain-trigger check (current vs. floor):
  overall              current=0.658 floor=0.494  OK (above floor)
  Buried               current=0.824 floor=0.516  OK (above floor)
  Overlooked           current=0.353 floor=0.213  OK (above floor)
  Long tail            current=0.671 floor=0.550  OK (above floor)
  Steady performers    current=0.648 floor=0.469  OK (above floor)


## Acknowledgments & data credit

*Mirrors the deployed paper's closing section — see `docs/index.html#acknowledgments`.*

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). Data credit and gratitude to FlyRank for the underlying warehouse this work is built on.

Thanks to Mirza, the mentor for this track, for the framing that shaped every stage of this work — from the initial research question through the honest-claims pass on these results. Special thanks to Alen for the opportunity to take part in this internship in the first place and to the whole FlyRank team that made this experience possible.

This paper was developed with the assistance of Claude (Anthropic's AI assistant), used for methodology review, drafting support, and revision passes across the capstone. Every finding, decision, and conclusion in this paper is my own.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Four new figures, built from the published result tables above (the heavy per-row intermediates in `work/outputs/*.parquet` and `*.csv` are gitignored — re-run `w05_model.ipynb → w06_validation_audit.ipynb → w07_action_playbook.ipynb` to regenerate the underlying numbers from scratch). Palette matches `work/figures/archetype_retention_vs_chance.png` and `tier_sizes.png` from ML-10 exactly, so all six figures the paper embeds read as one system.

Saved to `work/figures/`, then copied alongside the two ML-10 figures into `docs/figures/` for the deployed paper.

In [7]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIGD = "../figures"
DOCS_FIGD = "../../docs/figures"
os.makedirs(FIGD, exist_ok=True)
os.makedirs(DOCS_FIGD, exist_ok=True)

# Palette reused unchanged from work/notebooks/w07_action_playbook.ipynb (ML-10).
BLUE, GRAY, INK, SEC_INK, MUTED, GRID, BASE, SURF = \
    "#2a78d6", "#898781", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
TIER_RAMP = ["#1c5cab", "#2a78d6", "#5598e7", "#86b6ef"]  # ordinal, dark -> light

def style_axes(ax):
    ax.set_facecolor(SURF)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(BASE)
    ax.tick_params(colors=MUTED, length=0)
    ax.set_axisbelow(True)

print(f"figure setup ready -> {FIGD} / {DOCS_FIGD}")

figure setup ready -> ../figures / ../../docs/figures


In [8]:
# --- Figure: O/E forest plot -- the paper's key chart ---
oe_rows = [
    ("RULE: snippet_fix (ML-07)",     1.0319, 0.9248, 1.1642),
    ("MODEL: Steady performers",      1.0590, 0.8653, 1.2332),
    ("MODEL: Long tail",              1.0409, 0.9296, 1.1501),
    ("MODEL: Overlooked (all)",       0.8956, 0.7137, 1.0499),
    ("MODEL: Overlooked, top 1000",   0.7675, 0.6402, 0.9115),
    ("MODEL: Buried",                 0.6452, 0.4560, 0.8099),
]
oe_rows = sorted(oe_rows, key=lambda r: r[1])  # ascending O/E

fig, ax = plt.subplots(figsize=(7.5, 4.9), facecolor=SURF)
style_axes(ax)
for i, (label, oe, lo, hi) in enumerate(oe_rows):
    below_chance = hi < 1.0
    color = TIER_RAMP[0] if below_chance else GRAY
    ax.plot([lo, hi], [i, i], color=color, linewidth=2.2, solid_capstyle="round", zorder=2)
    ax.plot(oe, i, "o", color=color, markersize=8, zorder=3)
    ax.text(hi + 0.03, i, f"{oe:.2f}", va="center", ha="left", fontsize=9.5, color=INK)

ax.set_ylim(-0.6, len(oe_rows) - 0.3)
ax.axvline(1.0, color=SEC_INK, linewidth=1.2, linestyle=(0, (4, 3)), zorder=1)

ax.set_yticks(range(len(oe_rows)))
ax.set_yticklabels([r[0] for r in oe_rows], fontsize=10, color=SEC_INK)
ax.set_xlabel("Observed / Expected improve-rate (95% CI, bootstrapped by client)", color=SEC_INK)
ax.set_xlim(0.3, 1.5)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_title("Controlling for starting CTR erases the rule's apparent lift", color=INK, fontsize=12, pad=16)

legend_handles = [
    plt.Line2D([0], [0], color=TIER_RAMP[0], marker="o", linewidth=2, label="CI entirely below chance"),
    plt.Line2D([0], [0], color=GRAY, marker="o", linewidth=2, label="CI crosses chance"),
    plt.Line2D([0], [0], color=SEC_INK, linewidth=1.2, linestyle=(0, (4, 3)), label="chance (O/E = 1.0)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False, fontsize=8.8,
           bbox_to_anchor=(0.5, -0.03))
fig.text(0.01, -0.14,
    "Only Buried and Overlooked's top-1000 clear chance -- both below 1.0, not above.\n"
    "The rule's raw 1.14x lift does not survive CTR-adjustment.",
    fontsize=9.5, color=SEC_INK, ha="left", va="top")
fig.tight_layout()
oe_forest_path = f"{FIGD}/oe_forest.png"
fig.savefig(oe_forest_path, dpi=150, facecolor=SURF, bbox_inches="tight")
plt.close(fig)
print(f"wrote {oe_forest_path}")

wrote ../figures/oe_forest.png


In [9]:
# --- Figure: the regression-to-the-mean confound ---
strata_labels = ["0.000", "0.000-\n0.085", "0.085-\n0.14", "0.14-\n0.20", "0.20-\n0.27",
                  "0.27-\n0.37", "0.37-\n0.50", "0.50-\n0.75", "0.75+"]
strata_improve_rate = [0.2609, 0.4960, 0.4013, 0.3508, 0.3105, 0.2868, 0.2205, 0.2217, 0.1694]
base_rate = 0.2925

fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor=SURF)
style_axes(ax)
x = np.arange(len(strata_labels))
ax.bar(x, strata_improve_rate, color=BLUE, width=0.62, zorder=2)
ax.axhline(base_rate, color=SEC_INK, linewidth=1.2, linestyle=(0, (4, 3)), zorder=1)
ax.text(len(x) - 0.4, base_rate + 0.014, f"overall base rate {base_rate:.0%}",
        ha="right", va="bottom", fontsize=9, color=SEC_INK)

ax.set_xticks(x)
ax.set_xticklabels(strata_labels, fontsize=8.5, color=SEC_INK)
ax.set_xlabel("March CTR stratum", color=SEC_INK)
ax.set_ylabel("April improve-rate", color=SEC_INK)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.yaxis.grid(True, color=GRID, linewidth=0.8)

ax.set_title("Low-CTR pages are more likely to \"improve\" -- with no rule or model involved",
             color=INK, fontsize=11.5, pad=14)
fig.text(0.01, -0.08,
    "Improve-rate falls from ~50% to ~17% as starting CTR rises across the population.\n"
    "This is the confound the O/E-vs-chance figure controls for.",
    fontsize=9.5, color=SEC_INK, ha="left", va="top")
fig.tight_layout()
strata_path = f"{FIGD}/improve_by_ctr_stratum.png"
fig.savefig(strata_path, dpi=150, facecolor=SURF, bbox_inches="tight")
plt.close(fig)
print(f"wrote {strata_path}")

wrote ../figures/improve_by_ctr_stratum.png


In [10]:
# --- Figure: archetype centroid profile -- what "Buried" or "Overlooked" means, made visual ---
# Out-of-fold centroid medians, original units. From work/notebooks/w05_model.ipynb.
# Archetype identity is nominal categorical (not ordinal), so it gets a fixed-order,
# CVD-validated categorical palette (dataviz skill, palette.md slots 1-4) instead of a
# single-hue light->dark ramp -- a ramp reads as magnitude, not "which archetype is this."
# Validated: node scripts/validate_palette.js "#2a78d6,#eb6834,#1baf7a,#eda100" --mode light
# and the dark-step equivalents --mode dark -- both pass all four checks (adjacent pairs).
archetype_order = ["Buried", "Overlooked", "Steady performers", "Long tail"]  # tier priority order
archetype_color = dict(zip(archetype_order, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))

centroid_medians = {
    # archetype: (median_impressions, median_clicks, median_ctr, median_position)
    "Buried":            (260.0,   0.0, 0.000, 41.422),
    "Overlooked":        (498.5,   6.0, 1.172,  6.352),
    "Steady performers": (4388.5, 10.0, 0.277,  5.736),
    "Long tail":         (469.0,   0.0, 0.000,  8.225),
}
panels = [
    ("Median impressions\n(log scale)", 0, True,  lambda v: f"{v:,.0f}"),
    ("Median clicks",                   1, False, lambda v: f"{v:,.0f}"),
    ("Median CTR (%)",                  2, False, lambda v: f"{v:.2f}"),
    ("Median position\n(lower = better)", 3, False, lambda v: f"{v:.1f}"),
]

fig, axes = plt.subplots(1, 4, figsize=(11.5, 3.8), facecolor=SURF)
for ax, (title, idx, logscale, fmt) in zip(axes, panels):
    style_axes(ax)
    vals = [centroid_medians[a][idx] for a in archetype_order]
    colors = [archetype_color[a] for a in archetype_order]
    bars = ax.bar(range(4), vals, color=colors, width=0.65)
    # Direct value labels double as the contrast relief the palette check calls for
    # (two of the four hues sit below 3:1 on this light surface).
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), fmt(v),
                ha="center", va="bottom", fontsize=8, color=INK)
    ax.set_title(title, fontsize=9.5, color=INK)
    ax.set_xticks([])
    if logscale:
        ax.set_yscale("symlog")
    ax.tick_params(axis="y", colors=MUTED)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=archetype_color[a]) for a in archetype_order]
fig.legend(legend_handles, archetype_order, loc="lower center", ncol=4, frameon=False,
           fontsize=9.5, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("What each archetype looks like: search-side medians, out-of-fold",
             x=0.01, ha="left", fontsize=12, color=INK, y=1.06)
fig.text(0.01, -0.18,
    "Buried sits far below page 1 with zero clicks at the median; Overlooked earns a CTR ~3x\n"
    "anyone else's off modest volume; Steady performers carry the highest volume; Long tail is\n"
    "half the inventory at near-zero engagement.",
    fontsize=9.5, color=SEC_INK, ha="left", va="top")
fig.tight_layout()
profile_path = f"{FIGD}/archetype_profile.png"
fig.savefig(profile_path, dpi=150, facecolor=SURF, bbox_inches="tight")
plt.close(fig)
print(f"wrote {profile_path}")

wrote ../figures/archetype_profile.png


In [11]:
# --- Figure: May coverage funnel -- from work/notebooks/w07_action_playbook.ipynb ---
funnel_steps = [
    ("May raw rows",                        389_153),
    ("After dim_content match",             389_153 - 57_716),
    ("After publish/delete filter",         389_153 - 57_716 - 10_331),
    ("After impressions/position filter",   389_153 - 57_716 - 10_331 - 150_729),
    ("Queue-eligible (after volume floor)", 93_728),
]
assert funnel_steps[-2][1] - 76_649 == funnel_steps[-1][1]  # the volume-floor drop, checked

def fmt_k(v):
    """389153 -> '389.2k', 170377 -> '170.4k', 50000 -> '50k' -- whole thousands drop the decimal."""
    k = round(v / 1000, 1)
    return f"{k:.0f}k" if k == int(k) else f"{k:.1f}k"

labels = [s[0] for s in funnel_steps]
values = [s[1] for s in funnel_steps]
colors = [BASE, BASE, BASE, BASE, TIER_RAMP[0]]

fig, ax = plt.subplots(figsize=(7.5, 4), facecolor=SURF)
style_axes(ax)
ax.barh(range(len(labels)), values, color=colors, height=0.6, zorder=2)
for i, v in enumerate(values):
    ax.text(v + 4000, i, fmt_k(v), va="center", ha="left", fontsize=9.5, color=INK)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10, color=SEC_INK)
ax.invert_yaxis()
ax.set_xlabel("Pages", color=SEC_INK)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: fmt_k(v)))

ax.set_title("24.1% of the raw May inventory reaches the ranked action queue", color=INK, fontsize=12, pad=14)
fig.text(0.01, -0.09,
    "The rest (75.9%) is out of scope this cycle -- no March metadata match, unpublished,\n"
    "no search signal at all, or below the 100-impression volume floor.",
    fontsize=9.5, color=SEC_INK, ha="left", va="top")
fig.tight_layout()
funnel_path = f"{FIGD}/coverage_funnel.png"
fig.savefig(funnel_path, dpi=150, facecolor=SURF, bbox_inches="tight")
plt.close(fig)
print(f"wrote {funnel_path}")

wrote ../figures/coverage_funnel.png


In [12]:
# --- Copy all six figures (2 committed from ML-10 + 4 new) into docs/figures/ so the
# deployed paper (docs/index.html) can embed them with relative paths ---
existing_figures = ["archetype_retention_vs_chance.png", "tier_sizes.png"]
new_figures = ["oe_forest.png", "improve_by_ctr_stratum.png", "archetype_profile.png", "coverage_funnel.png"]

for name in existing_figures + new_figures:
    src = os.path.join(FIGD, name)
    assert os.path.exists(src), f"missing figure: {src}"
    shutil.copy(src, os.path.join(DOCS_FIGD, name))

print(f"copied {len(existing_figures) + len(new_figures)} figures to {DOCS_FIGD}")

copied 6 figures to ../../docs/figures


## 8. Telling the story

*The same work, retold for three audiences. The case study itself isn't here — it lives above, in the Question and Ranked recommendations sections, and in the deployed paper.*

### 5-minute demo outline

*For the Week-8 showcase. Timings are budgets, not targets — the honest result is the part worth protecting if I run long.*

**Question — 0:30.** FlyRank's content teams work a portfolio of hundreds of thousands of live search pages, and only a small fraction can get editorial attention each cycle. So: which pages are worth it? A CTR-based rule already answers that. I asked whether an unsupervised clustering answers it better — and whether either one beats doing nothing.

**Method — 1:00.** One month (March) of FlyRank warehouse search data, 331,437 content items aggregated from 9.8M daily rows. K-Means, k=4, on four search-side features only — impressions, clicks, CTR, position — never semantic or text. Validated with 5-fold GroupKFold by client, so no client's pages sit on both sides of a split. Both the rule and the clustering are then scored against the same April outcome window, which nothing in development ever touched.

**One chart — 1:30.** `docs/figures/oe_forest.png`, read left to right. Every bar is one group's observed-vs-expected improve rate, with the dashed line at chance. The point to land: the rule's raw 1.14× lift *looks* real until you adjust for each page's starting CTR — then it sits at O/E 1.03, CI [0.925, 1.164], straddling chance. Low-CTR pages improve more often on their own; the rule was mostly selecting them. That's regression to the mean, not rule skill.

**One honest result — 1:30.** Three of the four archetypes land at chance too — this is largely a null result, and I'd say so on stage. The exception is **Buried**: O/E 0.645, CI [0.456, 0.810], independently recomputed at 0.667 in a separate audit, and the most durable label across months at 82.4% March→May retention against 20.7% expected by chance. It's the only group where two independent signals agree — and note it points *down*: Buried pages improve *less* than expected, which is what makes them prune/rewrite candidates rather than a win.

**One recommendation — 0:30.** Route the 20,764 May pages labeled Buried into a prune/rewrite review queue, capped at 20 pages per client per cycle, with a named human signing off on every action. This is decision support for a monthly review, not automation and not a causal claim — the paper's no-go list says exactly where it must not be used.

### Two shareable cuts

**Social post — about the methodology.** *(Chart to attach: `docs/figures/oe_forest.png`.)*

> The most useful thing I built this internship wasn't the model. It was the control that killed my own baseline.
>
> The task: decide which of a content team's pages are worth an editor's limited time. A rule already existed — flag pages whose CTR sits below their position peers. It looked good: flagged pages improved 33.4% of the time against a 29.3% base rate, a 1.14× lift.
>
> Then I standardized. Low-CTR pages drift upward on their own, so I re-weighted each group's expected improve-rate to its own starting-CTR mix and asked observed ÷ expected instead. The rule landed at O/E 1.03, 95% CI [0.92, 1.16]. Chance. The lift was regression to the mean the whole time.
>
> Method in one line: K-Means (k=4) on four search-side signals only — never text — validated with 5-fold GroupKFold by client, then scored against an outcome month nothing in development had touched.
>
> One thing survived: a cluster I named "Buried" improves *less* often than expected (O/E 0.645, CI [0.46, 0.81]) and keeps its label across months 82.4% of the time against 20.7% by chance. Two independent signals, one group of 20,764 pages, and the finding points down rather than up.
>
> Most of this is a null result. Publishing it as the headline was the point.
>
> Paper and notebooks: https://jerovernay.github.io/FlyRank-Internship/

**Employer-facing summary — 3 sentences.**

> I built an unsupervised K-Means pipeline that sorts a content inventory into four performance archetypes and routes each to an editorial action, validated with GroupKFold-by-client cross-validation, a positive-control leakage probe, and a held-out outcome month. It runs on real production search data from FlyRank's warehouse — 331,437 content items aggregated from 9.8 million daily fact rows across 44 client accounts. It showed that the existing CTR-based triage rule's apparent 14% lift was regression to the mean rather than skill (O/E 1.03, CI crossing chance), while one archetype — durable at 82.4% month over month and improving measurably less often than expected — is the single group the evidence supports acting on.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
